# nb08_pack : 제출용 모델 묶음 만들기 (E4·E5 가중치 + P1 fold 모델)

- **가속기 : None (CPU)** · **인터넷 : ON** (kagglehub로 Nb04_train 버전별 Output 다운로드)
- **입력** : Dataset `hms-src`, 노트북 `Nb03` Output (profile/profile_features.csv, profile/p1_oof.csv)
- **출력** : `/kaggle/working/models/` → nb05(인터넷 OFF)에서 이 노트북 Output 하나만 연결하면 됨
  - `models/E4/`, `models/E5/` : fold0~4.pt, metrics.json, oof.csv (stage1 가중치 제외)
  - `models/P1/` : fold0~4.txt (LightGBM), p1_meta.json (입력 열 순서·반복 수), oof.csv
  - `e8_recipe.json` : 제출 조합 (nb07 섹션 6 선택 결과)
- **왜 필요한가** : 제출 노트북은 인터넷을 쓸 수 없고, 같은 노트북(Nb04_train)의 서로 다른 버전(E4=v6, E5=v7)을 동시에 Input으로 연결하기 어려움 → 한 곳에 모음

| 섹션 | 내용 |
|---|---|
| 0 | 설정 |
| 1 | 딥러닝 가중치 수집 (kagglehub, 버전 자동 탐색) |
| 2 | P1 재학습·저장 (nb03과 같은 설정) + 기존 p1_oof.csv와 재현 비교 |
| 3 | 제출 레시피 저장, 묶음 점검 |

## 0. 설정 (목적 : 입력 탐색·묶음 구성 지정)

In [ ]:
DL_EXPS = ["E4", "E5"]                 # 묶을 딥러닝 실험 (E8 레시피의 models)
RECIPE = {"candidate": "E4+E5 + P1v2", "models": DL_EXPS, "dl_weight": 0.85, "use_p1": True,
          "temperature": 1.0, "cv_kl_high_nested": 0.2955}      # nb07 v2 섹션 6 선택 결과 (10/23)
NB04_HANDLE = "sungik4057/nb04-train"
NB04_VERSIONS = range(20, 0, -1)       # 최신 버전부터 탐색
FETCH_TIMEOUT = 300                    # 버전 1개 다운로드 최대 대기(초)

import sys, os, json, time, shutil, signal, socket
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("HMS_INPUT_ROOT", "/kaggle/input"))
def find_file(*patterns):
    for pat in patterns:
        hits = sorted(INPUT_ROOT.glob(pat)) if INPUT_ROOT.exists() else []
        if hits:
            return hits[0]
    return None

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_file("**/src/eval/metrics.py").parents[2])
sys.path.insert(0, str(SRC_ROOT))
FEAT_PATH = Path(os.environ.get("HMS_FEATS") or find_file("**/profile/profile_features.csv"))
P1_OOF_PATH = Path(os.environ.get("HMS_P1_OOF") or find_file("**/profile/p1_oof.csv"))

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from src.config import load_config
from src.data.targets import PROB_COLS, CLASSES
from src.eval.metrics import evaluate

CFG = load_config(SRC_ROOT / "configs" / "base.yaml")
OUT = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "models"
OUT.mkdir(parents=True, exist_ok=True)
print("SRC_ROOT :", SRC_ROOT, "\nFEATS    :", FEAT_PATH, "\nP1 OOF   :", P1_OOF_PATH, "\nOUT      :", OUT)

## 1. 딥러닝 가중치 수집 (목적 : E4·E5 fold 가중치를 한 Output으로 모음)

- Input에 `models/<EXP>/fold0.pt`가 있으면 그것을 쓰고, 없으면 kagglehub로 Nb04_train 버전을 최신부터 확인
- Kaggle 안 kagglehub 기본 동작(Input 연결 후 무한 대기)을 피하려고 `DISABLE_KAGGLE_CACHE=1` + 시간 제한 사용

In [ ]:
class _FetchTimeout(Exception):
    pass

def _on_alarm(*_):
    raise _FetchTimeout()

def internet_on(host="www.kaggle.com", port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

def fetch_exp_dir(exp):
    """models/<exp>/ 폴더 경로 (Input 우선, 없으면 kagglehub로 해당 버전 Output 전체를 받음)."""
    hit = find_file(f"**/models/{exp}/fold0.pt")
    if hit is not None:
        print(f"{exp} : Input {hit.parent}")
        return hit.parent, None
    assert internet_on(), "인터넷 OFF : 설정에서 Internet ON으로 바꾸세요"
    os.environ["DISABLE_KAGGLE_CACHE"] = "1"
    import kagglehub
    t0 = time.time()
    for v in NB04_VERSIONS:
        signal.signal(signal.SIGALRM, _on_alarm)
        signal.alarm(60)
        try:                                                    # 먼저 작은 파일로 버전에 실험이 있는지 확인
            kagglehub.notebook_output_download(f"{NB04_HANDLE}/versions/{v}", path=f"models/{exp}/metrics.json")
        except Exception:
            continue
        finally:
            signal.alarm(0)
        files = [f"models/{exp}/fold{k}.pt" for k in range(5)] + [f"models/{exp}/oof.csv", f"models/{exp}/metrics.json"]
        signal.alarm(FETCH_TIMEOUT)
        try:
            got = [Path(kagglehub.notebook_output_download(f"{NB04_HANDLE}/versions/{v}", path=f)) for f in files]
        finally:
            signal.alarm(0)
        print(f"{exp} : kagglehub {NB04_HANDLE} v{v} | {len(got)}개 파일 | {time.time() - t0:.0f}s")
        return got[0].parent, v
    raise FileNotFoundError(f"{exp} : Nb04_train v{min(NB04_VERSIONS)}~v{max(NB04_VERSIONS)}에서 찾지 못함")

dl_versions = {}
for exp in DL_EXPS:
    src_dir, v = fetch_exp_dir(exp)
    dst = OUT / exp
    dst.mkdir(exist_ok=True)
    for name in [f"fold{k}.pt" for k in range(5)] + ["oof.csv", "metrics.json"]:
        shutil.copy2(src_dir / name, dst / name)
    with open(dst / "metrics.json") as f:
        mj = json.load(f)
    dl_versions[exp] = v
    print(f"  → {dst} | 입력 {mj['settings'].get('input', 'spec')} | 고득표 CV {mj['record']['cv_kl_high']} | "
          f"{sum(p.stat().st_size for p in dst.iterdir()) / 1e6:.0f}MB")

## 2. P1 재학습·저장 (목적 : 제출에서 쓸 P1 fold 모델 확보 / 판정 : 기존 OOF 재현)

- nb03 섹션 3과 **같은 설정** (soft label 행 복제 + 확률 가중치, 학습 fold 안 환자 10%로 조기 종료, 같은 시드)
- 입력 : nb03이 저장한 profile_features.csv (지표 재계산 없음)
- **재현 점검** : 새로 학습한 OOF와 nb03의 p1_oof.csv 고득표 KL 차이 0.005 이내면 통과 (LightGBM 스레드 수에 따라 미세한 차이 가능)

In [ ]:
feats = pd.read_csv(FEAT_PATH)
META_COLS = ["eeg_id", "patient_id", "fold", "high_vote", "expert_consensus"] + PROB_COLS + ["axis_lat", "axis_pd", "axis_mask", "y_seizure_bin"]
QC = ["nan_ratio", "n_flat", "max_abs"]
FEATS = [c for c in feats.columns if c not in META_COLS + QC]       # nb03 저장 순서 그대로 (지표 64개)
X_COLS = FEATS + QC
PARAMS = dict(objective="multiclass", num_class=len(CLASSES), learning_rate=0.05, num_leaves=31,
              min_data_in_leaf=40, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, verbose=-1, num_threads=os.cpu_count() or 4, seed=CFG["seed"])
print(f"지표 {len(FEATS)}개 + QC {len(QC)}개 = 입력 {len(X_COLS)}열 | {len(feats):,}건")

def expand(df):
    P = df[PROB_COLS].to_numpy()
    r, c = np.nonzero(P > 0)
    return df[X_COLS].to_numpy()[r], c, P[r, c]

p1_dir = OUT / "P1"
p1_dir.mkdir(exist_ok=True)
oof, best_iter, t0 = np.zeros((len(feats), len(CLASSES))), {}, time.time()
for k in sorted(feats.fold.unique()):
    tr, va = feats[feats.fold != k], feats[feats.fold == k]
    in_idx, es_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=CFG["seed"]).split(tr, groups=tr.patient_id))
    Xi, yi, wi = expand(tr.iloc[in_idx]); Xe, ye, we = expand(tr.iloc[es_idx])
    booster = lgb.train(PARAMS, lgb.Dataset(Xi, yi, weight=wi), num_boost_round=2000,
                        valid_sets=[lgb.Dataset(Xe, ye, weight=we)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[va.index] = booster.predict(va[X_COLS].to_numpy(), num_iteration=booster.best_iteration)
    booster.save_model(str(p1_dir / f"fold{int(k)}.txt"), num_iteration=booster.best_iteration)
    best_iter[int(k)] = int(booster.best_iteration)
    print(f"  fold {k} | rounds {booster.best_iteration} | 저장 fold{int(k)}.txt", flush=True)

new = evaluate(feats, oof)
old_df = pd.read_csv(P1_OOF_PATH).set_index("eeg_id").loc[feats.eeg_id]
old = evaluate(feats, old_df[[f"pred_{c}" for c in CLASSES]].to_numpy())
diff = abs(new["kl_high"] - old["kl_high"])
print(f"P1 재학습 OOF 고득표 KL {new['kl_high']:.4f} vs nb03 p1_oof {old['kl_high']:.4f} → 차이 {diff:.4f} "
      f"→ {'재현 통과' if diff <= 0.005 else '재현 차이 큼 : 설정 확인 필요'} | {time.time() - t0:.0f}s")

pd.concat([feats[["eeg_id", "patient_id", "fold", "high_vote"] + PROB_COLS].reset_index(drop=True),
           pd.DataFrame(oof, columns=[f"pred_{c}" for c in CLASSES])], axis=1).to_csv(p1_dir / "oof.csv", index=False)
with open(p1_dir / "p1_meta.json", "w") as f:
    json.dump({"x_cols": X_COLS, "feats": FEATS, "qc": QC, "best_iteration": best_iter, "params": PARAMS,
               "cv_kl_high": round(new["kl_high"], 4), "nb03_cv_kl_high": round(old["kl_high"], 4)}, f, indent=2)

## 3. 제출 레시피·묶음 점검 (목적 : nb05가 읽을 구성 저장)

In [ ]:
RECIPE["dl_versions"] = dl_versions
with open(OUT.parent / "e8_recipe.json", "w") as f:
    json.dump(RECIPE, f, indent=2, ensure_ascii=False)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT.parent)}  {p.stat().st_size / 1e6:.1f}MB")
total = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e6
print(f"\n레시피 : {RECIPE}\n총 {total:.0f}MB → 이 노트북 Save Version 후 nb05 Input으로 연결")

## 다음 단계
- 공유할 결과 : 섹션 1 실험별 버전·용량 줄, 섹션 2 **재현 점검 줄**, 섹션 3 레시피 줄
- nb05 : Input에서 Nb04_train을 빼고 **이 노트북(nb08_pack) Output**을 연결 → `PRESET` 선택 후 제출